In [1]:
import pandas as pd
import numpy as np
import os

DATA_ROOT = "../input/rsna-2022-cervical-spine-fracture-detection"

train_path = os.path.join(DATA_ROOT, "train.csv")
df_train = pd.read_csv(train_path)

target_cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]
epsilon = 1e-3
alpha = 0.1  # reduced Laplace smoothing for less bias


def smoothed_mean(series, a=alpha):
    return (series.sum() + a) / (len(series) + 2 * a)


marginal_means = {
    col: np.clip(smoothed_mean(df_train[col]), epsilon, 1 - epsilon)
    for col in target_cols
}
col_probs = {f"C{i}": marginal_means[f"C{i}"] for i in range(1, 8)}
vertebra_probs = np.array([col_probs[f"C{i}"] for i in range(1, 8)])

patient_overall_marginal = marginal_means["patient_overall"]
patient_overall_product = np.clip(1 - np.prod(1 - vertebra_probs), epsilon, 1 - epsilon)

cond_means = {}
for i in range(1, 8):
    col = f"C{i}"
    cond_means[col] = {
        1: np.clip(
            smoothed_mean(df_train.loc[df_train["patient_overall"] == 1, col]),
            epsilon,
            1 - epsilon,
        ),
        0: np.clip(
            smoothed_mean(df_train.loc[df_train["patient_overall"] == 0, col]),
            epsilon,
            1 - epsilon,
        ),
    }

calibrated_probs = {}
for i in range(1, 8):
    col = f"C{i}"
    calibrated_probs[col] = (
        patient_overall_marginal * cond_means[col][1]
        + (1 - patient_overall_marginal) * cond_means[col][0]
    )

blended_probs = {
    f"C{i}": np.clip(
        (col_probs[f"C{i}"] + calibrated_probs[f"C{i}"]) / 2.0, epsilon, 1 - epsilon
    )
    for i in range(1, 8)
}


def compute_scaled_probs(base_probs, target_overall, eps=epsilon):
    """Scale all vertebra probabilities by a common factor so that the
    derived product‑based overall probability equals `target_overall`."""
    upper = min((1 - eps) / v if v > 0 else 1e6 for v in base_probs.values())
    lower = 0.0
    for _ in range(30):  # binary search for scaling factor
        mid = (lower + upper) / 2.0
        scaled = {k: np.clip(mid * v, eps, 1 - eps) for k, v in base_probs.items()}
        overall = 1 - np.prod([1 - scaled[k] for k in scaled])
        if overall > target_overall:
            upper = mid
        else:
            lower = mid
    final_scaled = {k: np.clip(mid * v, eps, 1 - eps) for k, v in base_probs.items()}
    return final_scaled, overall


scaled_probs, _ = compute_scaled_probs(col_probs, patient_overall_marginal)
scaled_calibrated, _ = compute_scaled_probs(calibrated_probs, patient_overall_marginal)
scaled_blended, _ = compute_scaled_probs(blended_probs, patient_overall_marginal)

rows = []
for _, row in df_train.iterrows():
    uid = row["StudyInstanceUID"]
    for col in target_cols:
        rows.append({"StudyInstanceUID": uid, "prediction_type": col, "true": row[col]})
df_long = pd.DataFrame(rows)


def log_loss(y_true, y_pred):
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


df_long["pred_marginal"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else col_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_product"] = df_long.apply(
    lambda r: (
        patient_overall_product
        if r["prediction_type"] == "patient_overall"
        else col_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_calibrated"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else calibrated_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_calibrated_product"] = df_long.apply(
    lambda r: (
        patient_overall_product
        if r["prediction_type"] == "patient_overall"
        else calibrated_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_scaled"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else scaled_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_calibrated_scaled"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else scaled_calibrated[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_blended"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else blended_probs[r["prediction_type"]]
    ),
    axis=1,
)
df_long["pred_blended_scaled"] = df_long.apply(
    lambda r: (
        patient_overall_marginal
        if r["prediction_type"] == "patient_overall"
        else scaled_blended[r["prediction_type"]]
    ),
    axis=1,
)

weights = (
    df_long["prediction_type"]
    .apply(lambda x: 5 if x == "patient_overall" else 1)
    .values
)


def weighted_mean(loss_vals):
    return np.sum(loss_vals * weights) / np.sum(weights)


loss_marginal = weighted_mean(log_loss(df_long["true"], df_long["pred_marginal"]))
loss_product = weighted_mean(log_loss(df_long["true"], df_long["pred_product"]))
loss_calibrated = weighted_mean(log_loss(df_long["true"], df_long["pred_calibrated"]))
loss_calibrated_product = weighted_mean(
    log_loss(df_long["true"], df_long["pred_calibrated_product"])
)
loss_scaled = weighted_mean(log_loss(df_long["true"], df_long["pred_scaled"]))
loss_calibrated_scaled = weighted_mean(
    log_loss(df_long["true"], df_long["pred_calibrated_scaled"])
)
loss_blended = weighted_mean(log_loss(df_long["true"], df_long["pred_blended"]))
loss_blended_scaled = weighted_mean(
    log_loss(df_long["true"], df_long["pred_blended_scaled"])
)

losses = {
    "marginal": loss_marginal,
    "product": loss_product,
    "calibrated": loss_calibrated,
    "calibrated_product": loss_calibrated_product,
    "scaled": loss_scaled,
    "calibrated_scaled": loss_calibrated_scaled,
    "blended": loss_blended,
    "blended_scaled": loss_blended_scaled,
}
best_scheme = min(losses, key=losses.get)

# Choose the base probabilities according to the best scheme found above
if best_scheme == "marginal":
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = col_probs
elif best_scheme == "product":
    patient_overall_prob = patient_overall_product
    vertebra_prob_dict = col_probs
elif best_scheme == "calibrated":
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = calibrated_probs
elif best_scheme == "calibrated_product":
    patient_overall_prob = patient_overall_product
    vertebra_prob_dict = calibrated_probs
elif best_scheme == "scaled":
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = scaled_probs
elif best_scheme == "calibrated_scaled":
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = scaled_calibrated
elif best_scheme == "blended":
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = blended_probs
else:  # blended_scaled
    patient_overall_prob = patient_overall_marginal
    vertebra_prob_dict = scaled_blended

print(
    f"Weighted log‑loss – marginal: {loss_marginal:.5f}, product: {loss_product:.5f}, "
    f"calibrated: {loss_calibrated:.5f}, calibrated+product: {loss_calibrated_product:.5f}, "
    f"scaled: {loss_scaled:.5f}, calibrated_scaled: {loss_calibrated_scaled:.5f}, "
    f"blended: {loss_blended:.5f}, blended_scaled: {loss_blended_scaled:.5f}"
)
print(
    f"Chosen scheme: {best_scheme} → patient_overall_prob = {patient_overall_prob:.5f}"
)

# -------------------------------------------------------------------------
# Additional lightweight calibration: search a uniform scaling factor for
# vertebra probabilities that minimizes the weighted log‑loss on the training data.
# This keeps the core logic untouched while nudging the score toward the target.
# -------------------------------------------------------------------------
factor_range = np.arange(0.5, 1.51, 0.01)
best_factor = 1.0
base_loss = losses[best_scheme]
best_factor_loss = base_loss
best_scaled_dict = vertebra_prob_dict  # fallback


def loss_with_dict(v_dict):
    preds = df_long.apply(
        lambda r: (
            patient_overall_prob
            if r["prediction_type"] == "patient_overall"
            else v_dict[r["prediction_type"]]
        ),
        axis=1,
    )
    return weighted_mean(log_loss(df_long["true"], preds))


for f in factor_range:
    scaled_dict = {
        k: np.clip(f * v, epsilon, 1 - epsilon) for k, v in vertebra_prob_dict.items()
    }
    cur_loss = loss_with_dict(scaled_dict)
    if cur_loss < best_factor_loss:
        best_factor_loss = cur_loss
        best_factor = f
        best_scaled_dict = scaled_dict

if best_factor != 1.0:
    vertebra_prob_dict = best_scaled_dict
    print(
        f"Applied uniform scaling factor {best_factor:.2f} to vertebra probs, "
        f"improving weighted loss from {base_loss:.5f} to {best_factor_loss:.5f}"
    )
else:
    print("No scaling factor improved the loss; kept original probabilities.")



Weighted log‑loss – marginal: 0.48535, product: 0.49438, calibrated: 0.48536, calibrated+product: 0.49439, scaled: 0.48788, calibrated_scaled: 0.48788, blended: 0.48535, blended_scaled: 0.48788
Chosen scheme: marginal → patient_overall_prob = 0.46538
No scaling factor improved the loss; kept original probabilities.


In [2]:
test_path = os.path.join(DATA_ROOT, "test.csv")
df_test = pd.read_csv(test_path)

assert {"row_id", "StudyInstanceUID", "prediction_type"}.issubset(df_test.columns)




In [3]:
def get_prob(row):
    pred_type = row["prediction_type"]
    if pred_type == "patient_overall":
        return patient_overall_prob
    else:
        return vertebra_prob_dict[pred_type]


df_test["fractured"] = df_test.apply(get_prob, axis=1)



In [4]:
submission_path = "submission.csv"
df_test[["row_id", "fractured"]].to_csv(submission_path, index=False)

print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
